# ESCI multi-task reranker training

This notebook trains the **multi-task** ESCI reranker with a shared encoder and three heads:

- **Task 1:** query–product ranking (regression to ESCI gain for nDCG).
- **Task 2:** 4-class ESCI prediction (E/S/C/I).
- **Task 3:** substitute detection (binary: is the product a substitute?).

It mirrors the CLI entrypoint `python -m src.training.train_multi_task_reranker --config configs/multi_task_reranker.yaml` but runs everything inside the notebook.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
CHECKPOINTS_DIR = ROOT / "checkpoints"

In [ ]:
"""Configure and run multi-task training from YAML.

This loads `configs/multi_task_reranker.yaml`, overrides paths for the
notebook environment, and calls `MultiTaskTrainer`.
"""

import logging
import yaml

from src.training.train_multi_task_reranker import MultiTaskTrainer

logging.basicConfig(level=logging.INFO, format="%(message)s")

# Load config from YAML (matches configs/multi_task_reranker.yaml)
config_path = ROOT / "configs" / "multi_task_reranker.yaml"
with open(config_path) as f:
    cfg = yaml.safe_load(f) or {}

# Override paths for notebook (use absolute paths)
cfg["data_dir"] = str(DATA_DIR)
cfg["save_path"] = str(CHECKPOINTS_DIR / "multi_task_reranker")

# Keys accepted by MultiTaskTrainer
train_keys = {
    "data_dir",
    "model_name",
    "product_col",
    "save_path",
    "epochs",
    "batch_size",
    "max_length",
    "lr",
    "warmup_steps",
    "task_weight_ranking",
    "task_weight_esci",
    "task_weight_substitute",
    "evaluation_steps",
    "eval_max_queries",
    "small_version",
    "device",
    "val_frac",
    "recall_at",
}
train_cfg = {k: v for k, v in cfg.items() if k in train_keys}

trainer = MultiTaskTrainer(**train_cfg)
model = trainer.run()
model